# Bank Customer Churn Prediction (part 5 - ensembles and optuna)

Link to part 4 - https://github.com/Maxstef/data-loves-ml-for-people-course/blob/main/notebooks/2_4_ensembles_cross_val/0_2_bank_customer_searchcv.ipynb




## Imports & Files downloads

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np

from mlpeople.io.google_drive import download_file_iss

# download files from google drive
download_file_iss('1Xz-cqp6y-Y_oCiaSJeNsbU3-o5lM-5wY', output_path='./downloads/train.csv')
download_file_iss('1jsg2iPVphDMiHCEyqQeDSc4yDcMB3-r_', output_path='./downloads/test.csv')
download_file_iss('1tv0beG2n8cUQ6KcdaXhFzQI_Fd9640gg', output_path='./downloads/sample_submission.csv')

# read train.csv / show first 5 rows / show info
raw_df = pd.read_csv('downloads/train.csv', index_col=0)
target_col = "Exited"
raw_df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
id,,,,,,,,,,,,,
0,15779985.0,Nwankwo,678.0,France,Male,29.0,4.0,0.00,3.0,1.0,0.0,180626.36,0.0
1,15650086.0,Ch'in,687.0,France,Female,34.0,1.0,0.00,2.0,0.0,1.0,63736.17,0.0
2,15733602.0,Thompson,682.0,France,Female,52.0,6.0,0.00,3.0,0.0,0.0,179655.87,1.0
3,15645794.0,Macleod,753.0,Germany,Male,44.0,6.0,83347.25,2.0,1.0,0.0,161407.48,0.0
4,15633840.0,Hsia,544.0,Germany,Female,55.0,0.0,107747.57,1.0,1.0,0.0,176580.86,1.0


## Experiments Preparation

In [4]:
# raw data for experiments
X_raw = raw_df.drop([target_col, "CustomerId", "Surname"],axis=1)
y_raw = raw_df[target_col]

# previously it was determined that 20-40% is optimal for validataion data so n_splits=3 should be fine
skf = StratifiedKFold(n_splits=3, shuffle=False)

In [5]:
# Pipeline that includes previosly implemented preprocessing steps
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder

from mlpeople.preprocessing.transformers import (
    ColumnDropper,
    NumericBinaryFlagTransformer,
    CategoricalBinaryFlagTransformer,
    TopNCategoricalTransformer,
    NumericBinner,
)

numeric_selector = make_column_selector(dtype_include=np.number)
categorical_selector = make_column_selector(dtype_exclude=np.number)

numeric_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("poly_features", PolynomialFeatures(include_bias=False))
])

categorical_pipeline = Pipeline(
    [("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="if_binary"))]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, numeric_selector),
        ("cat", categorical_pipeline, categorical_selector),
    ]
)

# default (example) pipeline
pipeline_base = Pipeline([
    ("drop_cols", ColumnDropper([])),
    ("num_flags", NumericBinaryFlagTransformer({})),
    ("num_bin", NumericBinner({})),
    ("cat_flags", CategoricalBinaryFlagTransformer({})),
    ("top_n", TopNCategoricalTransformer({})),
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(solver='liblinear'))
])

In [6]:
# helper function to display cv search result
def display_cv_result(
    cv_results,
    view_params=[],
    view_count=10
):
    result_df = pd.DataFrame(cv_results)
    view_params.extend([
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ])
    result_df_view = result_df[
        view_params
    ].sort_values("mean_test_score", ascending=False)

    with pd.option_context('display.max_colwidth', None):
        display(
            result_df_view.head(view_count)
        )


## Best previous

In [7]:
from math import inf

# best config for CategoricalBinaryFlagTransformer
cat_flag_is_germany = {
    "flag_name": "IsGermany",
    "value": "Germany",
    "drop_original": True
}
cat_flags_config_default = {
    "Geography": [cat_flag_is_germany]
}
# for further experiments
cat_flag_is_spain = {
    "flag_name": "IsSpain",
    "value": "Spain",
    "drop_original": True
}
cat_flag_is_france = {
    "flag_name": "IsFrance",
    "value": "France",
    "drop_original": True
}

# best config for NumericBinner transformer
num_bin_salary = {
    "bins": [0, 25000, 100000, inf],
    "labels": ["low", "medium", "high"],
    "new_col": "SalaryScore",
    "drop_original": True
}
num_bin_credit_score = {
    "bins": [0, 675, inf],
    "labels": ["low", "high"],
    "new_col": "CreditScoreScore",
    "drop_original": True
}
num_bin_mapping_default = {
    "EstimatedSalary": num_bin_salary,
    "CreditScore": num_bin_credit_score,
}

pipeline_prev_best = Pipeline([
    ("drop_cols", ColumnDropper(["Tenure", "HasCrCard"])),
    ("num_flags", NumericBinaryFlagTransformer({})),
    ("num_bin", NumericBinner(num_bin_mapping_default)),
    ("cat_flags", CategoricalBinaryFlagTransformer(cat_flags_config_default)),
    ("top_n", TopNCategoricalTransformer({})),
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(solver='liblinear'))
])

In [8]:
grid_search_prev_best = GridSearchCV(
    pipeline_prev_best,
    {
        "drop_cols__columns": [
            ["Tenure", "HasCrCard"],
        ],
        "preprocessing__num__poly_features__degree": [3],
        "model__C": [0.3, 0.5],
        "model__l1_ratio": [1.0],
    },
    cv=skf,
    scoring="roc_auc",
)

grid_search_prev_best.fit(X_raw, y_raw)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'drop_cols__columns': [['Tenure', 'HasCrCard']], 'model__C': [0.3, 0.5], 'model__l1_ratio': [1.0], 'preprocessing__num__poly_features__degree': [3]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"verbose verbose: intControls the verbosity: the higher, the more me

In [9]:
display_cv_result(
    grid_search_prev_best.cv_results_,
    view_params=[
        "param_model__l1_ratio",
        "param_model__C",
    ]
)

,param_model__l1_ratio,param_model__C,mean_test_score,std_test_score,rank_test_score
0,1.0,0.3,0.936473,0.000665,1
1,1.0,0.5,0.936454,0.000765,2


In [10]:
best_lg_model_pipeline = grid_search_prev_best.best_estimator_
best_lg_model_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['Tenure', 'HasCrCard']"
,flag_config,{}
,mapping,"{'CreditScore': {'bins': [0, 675, ...], 'drop_original': True, 'labels': ['low', 'high'], 'new_col': 'CreditScoreScore'}, 'EstimatedSalary': {'bins': [0, 25000, ...], 'drop_original': True, 'labels': ['low', 'medium', ...], 'new_col': 'SalaryScore'}}"
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outpu

## Random Forest

In [11]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

# start with previous best pipelein copy
pipeline_rf = clone(pipeline_prev_best)

pipeline_rf.set_params(
    model=RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
    )
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['Tenure', 'HasCrCard']"
,flag_config,{}
,mapping,"{'CreditScore': {'bins': [0, 675, ...], 'drop_original': True, 'labels': ['low', 'high'], 'new_col': 'CreditScoreScore'}, 'EstimatedSalary': {'bins': [0, 25000, ...], 'drop_original': True, 'labels': ['low', 'medium', ...], 'new_col': 'SalaryScore'}}"
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outpu

### Drop cols & Poly degree & Custom flags

In [12]:
%%time

# some combinations of param options might cause error
import warnings
warnings.filterwarnings('ignore')

cat_flags_config_all_options = [
    {}, 
    cat_flags_config_default,
    {
        "Geography": [cat_flag_is_germany, cat_flag_is_france]
    },
    {
        "Geography": [cat_flag_is_germany, cat_flag_is_spain]
    },
    {
        "Geography": [cat_flag_is_germany, cat_flag_is_spain, cat_flag_is_france]
    },
]

num_bin_mapping_all_options = [
    {},
    num_bin_mapping_default,
    {
        "EstimatedSalary": num_bin_salary,
    },
    {
        "CreditScore": num_bin_credit_score,
    }
]


random_search_rf_drop_poly = RandomizedSearchCV(
    pipeline_rf,
    {
        "drop_cols__columns": [
            [],
            ["Tenure"],
            ["Tenure", "HasCrCard"],
            ["Tenure", "HasCrCard", "EstimatedSalary"],
            ["Tenure", "HasCrCard", "CreditScore"],
            ["Tenure", "HasCrCard", "EstimatedSalary", "CreditScore"],
        ],
        "preprocessing__num__poly_features__degree": [1, 2, 3],
        "preprocessing__cat__encoder__drop": ["if_binary", None],
        "cat_flags__flag_config": cat_flags_config_all_options,
        "num_bin__mapping": num_bin_mapping_all_options,
    },
    cv=skf,
    scoring="roc_auc",
    n_iter=80,
)

random_search_rf_drop_poly.fit(X_raw, y_raw)

CPU times: user 2min 57s, sys: 4.79 s, total: 3min 2s
Wall time: 33.4 s


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'cat_flags__flag_config': [{}, {'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}]}, ...], 'drop_cols__columns': [[], ['Tenure'], ...], 'num_bin__mapping': [{}, {'CreditScore': {'bins': [0, 675, ...], 'drop_original': True, 'labels': ['low', 'high'], 'new_col': 'CreditScoreScore'}, 'EstimatedSalary': {'bins': [0, 25000, ...], 'drop_original': True, 'labels': ['low', 'medium', ...], 'new_col': 'SalaryScore'}}, ...], 'preprocessing__cat__encoder__drop': ['if_binary', None], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",80
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) spli

In [13]:
display_cv_result(
    random_search_rf_drop_poly.cv_results_,
    view_params=[
        "param_drop_cols__columns",
        "param_preprocessing__num__poly_features__degree",
        "param_preprocessing__cat__encoder__drop",
        "param_cat_flags__flag_config",
        "param_num_bin__mapping",
    ],
    view_count=15,
)

,param_drop_cols__columns,param_preprocessing__num__poly_features__degree,param_preprocessing__cat__encoder__drop,param_cat_flags__flag_config,param_num_bin__mapping,mean_test_score,std_test_score,rank_test_score
59,[],2,None,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}]}",{},0.923305,0.000776,1
27,[],1,if_binary,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsSpain', 'value': 'Spain', 'drop_original': True}]}",{},0.923280,0.000819,2
52,[],2,None,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsSpain', 'value': 'Spain', 'drop_original': True}]}",{},0.922035,0.001284,3
39,"[Tenure, HasCrCard]",2,None,{},{},0.921869,0.001219,4
25,[],2,if_binary,{},{},0.921361,0.001271,5
26,[],1,None,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsSpain', 'value': 'Spain', 'drop_original': True}, {'flag_name': 'IsFrance', 'value': 'France', 'drop_original': True}]}",{},0.921346,0.001504,6
30,"[Tenure, HasCrCard]",1,None,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsSpain', 'value': 'Spain', 'drop_original': True}, {'flag_name': 'IsFrance', 'value': 'France', 'drop_original': True}]}",{},0.921273,0.000704,7
50,[Tenure],1,if_binary,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsFrance', 'value': 'France', 'drop_original': True}]}",{},0.921176,0.001384,8
40,[Tenure],1,None,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsFrance', 'value': 'France', 'drop_original': True}]}",{},0.921090,0.001967,9
8,"[Tenure, HasCrCard]",1,if_binary,"{'Geography': [{'flag_name': 'IsGermany', 'value': 'Germany', 'drop_original': True}, {'flag_name': 'IsFrance', 'value': 'France', 'drop_original': True}]}",{},0.920394,0.000717,10


In [14]:
# update random forest pipeline based on above result
pipeline_rf = Pipeline([
    ("drop_cols", ColumnDropper([])), # leave all cols
    ("num_flags", NumericBinaryFlagTransformer({})),
    ("num_bin", NumericBinner({})), # no custom num bin, leave cols as it is
    ("cat_flags", CategoricalBinaryFlagTransformer({
        "Geography": [cat_flag_is_germany, cat_flag_is_france] # add france here
    })),
    ("top_n", TopNCategoricalTransformer({})),
    ("preprocessing", preprocessor), # 2 degree is default for PolynomialFeatures
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
    ))
])

### Hyperparams tuning - RandomizedSearchCV

In [28]:
%%time

random_search_rf_hyper_tun = RandomizedSearchCV(
    pipeline_rf,
    {
        "model__n_estimators": range(10, 300, 10),
        "model__max_depth": range(3, 20, 2),
        "model__max_leaf_nodes": range(10, 2000, 10),
        "model__criterion": ["gini", "entropy"],
        "model__min_samples_split": range(2, 20),
        "model__min_samples_leaf": range(1, 10),
        "model__max_features": ["sqrt", "log2", 0.5, 0.8],
        "model__max_samples": [None, 0.6],
        "model__bootstrap": [True, False],
        "model__min_impurity_decrease": [0.0, 0.1],
        "model__class_weight": ["balanced", "balanced_subsample", {0.0: 1, 1.0: 5}, {0.0: 5, 1.0: 1}],
    },
    cv=skf,
    scoring="roc_auc",
    n_iter=80,
    random_state=42,
)

random_search_rf_hyper_tun.fit(X_raw, y_raw)

CPU times: user 17min 49s, sys: 20.5 s, total: 18min 10s
Wall time: 2min 51s


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__bootstrap': [True, False], 'model__class_weight': ['balanced', 'balanced_subsample', ...], 'model__criterion': ['gini', 'entropy'], 'model__max_depth': range(3, 20, 2), ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",80
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the vari

In [29]:
display_cv_result(
    random_search_rf_hyper_tun.cv_results_,
    view_params=[
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__max_leaf_nodes",
        "param_model__criterion",
        "param_model__min_samples_split",
        "param_model__min_samples_leaf",
        "param_model__max_features",
        "param_model__max_samples",
        "param_model__bootstrap",
        "param_model__min_impurity_decrease",
        "param_model__class_weight",
    ],
    view_count=15,
)

,param_model__n_estimators,param_model__max_depth,param_model__max_leaf_nodes,param_model__criterion,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features,param_model__max_samples,param_model__bootstrap,param_model__min_impurity_decrease,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
70,160,7,1620,entropy,19,9,log2,None,True,0.0,"{0.0: 1, 1.0: 5}",0.930827,0.000872,1
17,210,7,240,gini,18,1,sqrt,None,True,0.0,balanced,0.930803,0.000647,2
23,230,11,1190,entropy,15,3,log2,0.6,True,0.0,balanced,0.930744,0.000813,3
62,80,9,1290,gini,12,8,log2,None,False,0.0,balanced_subsample,0.930413,0.000399,4
5,220,5,1640,entropy,7,3,0.5,0.6,True,0.0,balanced_subsample,0.930048,0.000903,5
65,110,9,1470,entropy,3,2,log2,None,False,0.0,balanced_subsample,0.929995,0.000863,6
77,50,5,690,gini,11,4,0.5,None,True,0.0,balanced,0.929704,0.001145,7
72,260,11,1910,gini,17,9,log2,0.6,True,0.0,balanced,0.929640,0.000819,8
75,190,9,460,gini,3,3,sqrt,None,True,0.0,"{0.0: 5, 1.0: 1}",0.929613,0.001387,9
31,130,19,170,gini,12,2,sqrt,None,False,0.0,"{0.0: 5, 1.0: 1}",0.929400,0.001112,10


In [38]:
# save best random forest classifier
best_pipeline_rf_rs = random_search_rf_hyper_tun.best_estimator_
best_pipeline_rf_rs

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,[]
,flag_config,{}
,mapping,{}
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}, {'drop_original': True, 'flag_name': 'IsFrance', 'value': 'France'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that 

### Hyperparams tuning - Optuna

#### Define Objective Function

In [ ]:
import optuna
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective_rf(trial):
    pipeline = clone(pipeline_rf)

    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Suggest hyperparameters
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 300, step=10),
        "max_depth": trial.suggest_int("max_depth", 3, 20, step=2),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 2000, step=10),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8]),
        "min_impurity_decrease": trial.suggest_categorical("min_impurity_decrease", [0.0, 0.1]),
        "class_weight": trial.suggest_categorical(
            "class_weight",
            [
                "balanced",
                "balanced_subsample",
                {0.0: 1, 1.0: 5},
                {0.0: 5, 1.0: 1},
            ],
        ),
        "bootstrap": bootstrap,
        "random_state": 42,
        "n_jobs": -1,
    }

    # Only suggest max_samples if bootstrap=True
    if bootstrap:
        params["max_samples"] = trial.suggest_categorical(
            "max_samples",
            [None, 0.9, 0.6, 0.3], # add more options than in RandomizedSearchCV
        )

    pipeline.set_params(model=RandomForestClassifier(**params))

    scores = cross_val_score(
        pipeline,
        X_raw,
        y_raw,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1,
    )

    return scores.mean()

#### Run Study

In [ ]:
%%time

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=50)

[I 2026-03-03 13:44:07,647] A new study created in memory with name: no-name-0b2b8e69-330d-487a-a2c8-0bc6d9e286b5
[I 2026-03-03 13:44:18,114] Trial 0 finished with value: 0.9213494014769926 and parameters: {'bootstrap': False, 'n_estimators': 150, 'max_depth': 7, 'max_leaf_nodes': 790, 'criterion': 'entropy', 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 0.8, 'min_impurity_decrease': 0.0, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9213494014769926.
[I 2026-03-03 13:44:19,500] Trial 1 finished with value: 0.8976635071979716 and parameters: {'bootstrap': True, 'n_estimators': 60, 'max_depth': 15, 'max_leaf_nodes': 100, 'criterion': 'entropy', 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 0.5, 'min_impurity_decrease': 0.1, 'class_weight': 'balanced_subsample', 'max_samples': 0.6}. Best is trial 0 with value: 0.9213494014769926.
[I 2026-03-03 13:44:20,989] Trial 2 finished with value: 0.9297543770820204 and parameters: {'bootstrap': True, '

CPU times: user 547 ms, sys: 338 ms, total: 885 ms
Wall time: 57.9 s


#### Inspect Results

In [ ]:
print("Best ROC AUC:", study_rf.best_value)
print("Best params:", study_rf.best_params)

Best ROC AUC: 0.9318455897569469
Best params: {'bootstrap': False, 'n_estimators': 170, 'max_depth': 7, 'max_leaf_nodes': 1180, 'criterion': 'entropy', 'min_samples_split': 6, 'min_samples_leaf': 9, 'max_features': 'log2', 'min_impurity_decrease': 0.0, 'class_weight': {0.0: 5, 1.0: 1}}


#### Train Final Model

In [43]:
best_pipeline_rf_opt = clone(pipeline_rf)
best_pipeline_rf_opt.set_params(
    model=RandomForestClassifier(
        **study_rf.best_params,
        random_state=42,
        n_jobs=-1,
    )
)

best_pipeline_rf_opt.fit(X_raw, y_raw)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,[]
,flag_config,{}
,mapping,{}
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}, {'drop_original': True, 'flag_name': 'IsFrance', 'value': 'France'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that 

#### Visualization

In [44]:
optuna.visualization.plot_optimization_history(study_rf)

In [45]:
optuna.visualization.plot_param_importances(study_rf)

## Adaboost

### Optuna Objective for AdaBoost

In [53]:
import optuna
import copy
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score


def objective_adaboost(trial):
    pipeline = clone(pipeline_rf)

    # =========================
    # STRUCTURE SEARCH
    # =========================

    drop_columns = trial.suggest_categorical(
        "drop_cols",
        [
            [],
            ["Tenure"],
            ["Tenure", "HasCrCard"],
        ],
    )

    poly_degree = trial.suggest_int("poly_degree", 1, 3)

    encoder_drop = trial.suggest_categorical(
        "encoder_drop",
        ["if_binary", None],
    )

    cat_flags_idx = trial.suggest_int(
        "cat_flags_idx",
        0,
        len(cat_flags_config_all_options) - 1,
    )

    num_bin_idx = trial.suggest_int(
        "num_bin_idx",
        0,
        len(num_bin_mapping_all_options) - 1,
    )

    pipeline.set_params(
        drop_cols__columns=drop_columns,
        preprocessing__num__poly_features__degree=poly_degree,
        preprocessing__cat__encoder__drop=encoder_drop,
        cat_flags__flag_config=copy.deepcopy(
            cat_flags_config_all_options[cat_flags_idx]
        ),
        num_bin__mapping=copy.deepcopy(
            num_bin_mapping_all_options[num_bin_idx]
        ),
    )

    # =========================
    # ADABOOST HYPERPARAMETERS
    # =========================

    # Base tree complexity
    tree_max_depth = trial.suggest_int("tree_max_depth", 1, 5)

    base_estimator = DecisionTreeClassifier(
        max_depth=tree_max_depth,
        min_samples_split=trial.suggest_int("tree_min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("tree_min_samples_leaf", 1, 10),
        random_state=42,
    )

    ada_params = {
        "estimator": base_estimator,
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "random_state": 42,
    }

    pipeline.set_params(model=AdaBoostClassifier(**ada_params))

    scores = cross_val_score(
        pipeline,
        X_raw,
        y_raw,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1,
    )

    return scores.mean()

### Run Study

In [55]:
%%time

study_ada = optuna.create_study(direction="maximize")
study_ada.optimize(objective_adaboost, n_trials=30)

[I 2026-03-03 14:03:23,930] A new study created in memory with name: no-name-f3630120-720e-43ca-a426-26e639dafc13
[I 2026-03-03 14:03:26,478] Trial 0 finished with value: 0.9305418729369791 and parameters: {'drop_cols': ['Tenure', 'HasCrCard'], 'poly_degree': 1, 'encoder_drop': None, 'cat_flags_idx': 2, 'num_bin_idx': 3, 'tree_max_depth': 3, 'tree_min_samples_split': 4, 'tree_min_samples_leaf': 1, 'n_estimators': 175, 'learning_rate': 0.020958719586349622}. Best is trial 0 with value: 0.9305418729369791.
[I 2026-03-03 14:03:28,414] Trial 1 finished with value: 0.9269272892270215 and parameters: {'drop_cols': [], 'poly_degree': 1, 'encoder_drop': None, 'cat_flags_idx': 4, 'num_bin_idx': 1, 'tree_max_depth': 5, 'tree_min_samples_split': 20, 'tree_min_samples_leaf': 7, 'n_estimators': 125, 'learning_rate': 0.5815684194034022}. Best is trial 0 with value: 0.9305418729369791.
[I 2026-03-03 14:04:00,685] Trial 2 finished with value: 0.9310216171493629 and parameters: {'drop_cols': [], 'poly_

CPU times: user 620 ms, sys: 612 ms, total: 1.23 s
Wall time: 7min 1s


In [56]:
print("Best ROC AUC:", study_ada.best_value)
print("Best params:", study_ada.best_params)

Best ROC AUC: 0.9359984979825331
Best params: {'drop_cols': ['Tenure'], 'poly_degree': 2, 'encoder_drop': 'if_binary', 'cat_flags_idx': 3, 'num_bin_idx': 0, 'tree_max_depth': 2, 'tree_min_samples_split': 7, 'tree_min_samples_leaf': 4, 'n_estimators': 400, 'learning_rate': 0.154391447204384}


### Best Model Pipeline

In [57]:
def build_best_adaboost_pipeline(study):
    best = study.best_params

    pipeline = clone(pipeline_rf)

    # =========================
    # STRUCTURE
    # =========================

    cat_flags_config = copy.deepcopy(
        cat_flags_config_all_options[best["cat_flags_idx"]]
    )

    num_bin_mapping = copy.deepcopy(
        num_bin_mapping_all_options[best["num_bin_idx"]]
    )

    pipeline.set_params(
        drop_cols__columns=best["drop_cols"],
        preprocessing__num__poly_features__degree=best["poly_degree"],
        preprocessing__cat__encoder__drop=best["encoder_drop"],
        cat_flags__flag_config=cat_flags_config,
        num_bin__mapping=num_bin_mapping,
    )

    # =========================
    # MODEL
    # =========================

    base_estimator = DecisionTreeClassifier(
        max_depth=best["tree_max_depth"],
        min_samples_split=best["tree_min_samples_split"],
        min_samples_leaf=best["tree_min_samples_leaf"],
        random_state=42,
    )

    model = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=best["n_estimators"],
        learning_rate=best["learning_rate"],
        random_state=42,
    )

    pipeline.set_params(model=model)

    return pipeline

In [58]:
best_pipeline_ada_opt = build_best_adaboost_pipeline(study_ada)
best_pipeline_ada_opt.fit(X_raw, y_raw)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,['Tenure']
,flag_config,{}
,mapping,{}
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}, {'drop_original': True, 'flag_name': 'IsSpain', 'value': 'Spain'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note

### Visualizations

In [59]:
optuna.visualization.plot_optimization_history(study_ada)

In [60]:
optuna.visualization.plot_param_importances(study_ada)

## XGBoost

In [61]:
import xgboost as xgb
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

### Define Objective

In [62]:
def objective_xgb(trial):
    pipeline = clone(pipeline_rf)

    # =========================
    # STRUCTURE SEARCH
    # =========================
    poly_degree = trial.suggest_int("poly_degree", 1, 3)

    encoder_drop = trial.suggest_categorical("encoder_drop", ["if_binary", None])

    cat_flags_idx = trial.suggest_int(
        "cat_flags_idx", 0, len(cat_flags_config_all_options) - 1
    )

    num_bin_idx = trial.suggest_int(
        "num_bin_idx", 0, len(num_bin_mapping_all_options) - 1
    )

    cat_flags_config = copy.deepcopy(cat_flags_config_all_options[cat_flags_idx])
    num_bin_mapping = copy.deepcopy(num_bin_mapping_all_options[num_bin_idx])

    # =========================
    # SMART DROP LOGIC
    # =========================
    protected_columns = set(num_bin_mapping.keys())

    ALL_DROP_OPTIONS = [
        [],
        ["Tenure"],
        ["Tenure", "HasCrCard"],
        ["Tenure", "HasCrCard", "EstimatedSalary"],
        ["Tenure", "HasCrCard", "CreditScore"],
        ["Tenure", "HasCrCard", "EstimatedSalary", "CreditScore"],
    ]

    drop_columns = trial.suggest_categorical("drop_cols", ALL_DROP_OPTIONS)

    # Prune if drop_columns conflicts with protected_columns
    if any(col in protected_columns for col in drop_columns):
        raise optuna.exceptions.TrialPruned()

    # =========================
    # APPLY STRUCTURE
    # =========================
    pipeline.set_params(
        drop_cols__columns=drop_columns,
        preprocessing__num__poly_features__degree=poly_degree,
        preprocessing__cat__encoder__drop=encoder_drop,
        cat_flags__flag_config=cat_flags_config,
        num_bin__mapping=num_bin_mapping,
    )

    # =========================
    # XGBOOST HYPERPARAMETERS
    # =========================
    xgb_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "use_label_encoder": False,
        "eval_metric": "auc",
        "n_jobs": -1,
    }

    pipeline.set_params(model=xgb.XGBClassifier(**xgb_params))

    # =========================
    # CROSS-VALIDATION
    # =========================
    scores = cross_val_score(
        pipeline,
        X_raw,
        y_raw,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1,
    )

    return scores.mean()

### Run Study

In [68]:
%%time

# suppress Optuna info logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=200)

CPU times: user 2.07 s, sys: 463 ms, total: 2.53 s
Wall time: 22.8 s


In [69]:
print("Best ROC AUC:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

Best ROC AUC: 0.9369650483120333
Best params: {'poly_degree': 1, 'encoder_drop': None, 'cat_flags_idx': 1, 'num_bin_idx': 1, 'drop_cols': ['Tenure'], 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.22545878046149048, 'subsample': 0.8100024200131671, 'colsample_bytree': 0.7373211078599748, 'gamma': 1.5327583557683286, 'reg_alpha': 4.946882393540392, 'reg_lambda': 4.11613448594875}


### Best Model Pipeline

In [70]:
def build_best_xgb_pipeline(study):
    best = study.best_params
    pipeline = clone(pipeline_rf)

    cat_flags_config = copy.deepcopy(cat_flags_config_all_options[best["cat_flags_idx"]])
    num_bin_mapping = copy.deepcopy(num_bin_mapping_all_options[best["num_bin_idx"]])

    pipeline.set_params(
        drop_cols__columns=best["drop_cols"],
        preprocessing__num__poly_features__degree=best["poly_degree"],
        preprocessing__cat__encoder__drop=best["encoder_drop"],
        cat_flags__flag_config=cat_flags_config,
        num_bin__mapping=num_bin_mapping,
    )

    xgb_params = {
        "n_estimators": best["n_estimators"],
        "max_depth": best["max_depth"],
        "learning_rate": best["learning_rate"],
        "subsample": best["subsample"],
        "colsample_bytree": best["colsample_bytree"],
        "gamma": best["gamma"],
        "reg_alpha": best["reg_alpha"],
        "reg_lambda": best["reg_lambda"],
        "random_state": 42,
        "use_label_encoder": False,
        "eval_metric": "auc",
        "n_jobs": -1,
    }

    pipeline.set_params(model=xgb.XGBClassifier(**xgb_params))
    return pipeline

best_pipeline_xgb_opt = build_best_xgb_pipeline(study_xgb)
best_pipeline_xgb_opt.fit(X_raw, y_raw)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,['Tenure']
,flag_config,{}
,mapping,"{'CreditScore': {'bins': [0, 675, ...], 'drop_original': True, 'labels': ['low', 'high'], 'new_col': 'CreditScoreScore'}, 'EstimatedSalary': {'bins': [0, 25000, ...], 'drop_original': True, 'labels': ['low', 'medium', ...], 'new_col': 'SalaryScore'}}"
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`

### Visualizations

In [80]:
optuna.visualization.plot_optimization_history(study_xgb)

In [82]:
optuna.visualization.plot_param_importances(study_xgb)

## LightGBM

In [71]:
import lightgbm as lgb
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

### Define LightGBM Objective

In [74]:
def objective_lgb(trial):
    pipeline = clone(pipeline_rf)

    # =========================
    # STRUCTURE SEARCH
    # =========================
    poly_degree = trial.suggest_int("poly_degree", 1, 3)
    encoder_drop = trial.suggest_categorical("encoder_drop", ["if_binary", None])

    cat_flags_idx = trial.suggest_int(
        "cat_flags_idx", 0, len(cat_flags_config_all_options) - 1
    )

    num_bin_idx = trial.suggest_int(
        "num_bin_idx", 0, len(num_bin_mapping_all_options) - 1
    )

    cat_flags_config = copy.deepcopy(cat_flags_config_all_options[cat_flags_idx])
    num_bin_mapping = copy.deepcopy(num_bin_mapping_all_options[num_bin_idx])

    # =========================
    # SMART DROP LOGIC
    # =========================
    protected_columns = set(num_bin_mapping.keys())

    ALL_DROP_OPTIONS = [
        [],
        ["Tenure"],
        ["Tenure", "HasCrCard"],
        ["Tenure", "HasCrCard", "EstimatedSalary"],
        ["Tenure", "HasCrCard", "CreditScore"],
        ["Tenure", "HasCrCard", "EstimatedSalary", "CreditScore"],
    ]

    drop_columns = trial.suggest_categorical("drop_cols", ALL_DROP_OPTIONS)

    # Prune invalid configs
    if any(col in protected_columns for col in drop_columns):
        raise optuna.exceptions.TrialPruned()

    # =========================
    # APPLY STRUCTURE
    # =========================
    pipeline.set_params(
        drop_cols__columns=drop_columns,
        preprocessing__num__poly_features__degree=poly_degree,
        preprocessing__cat__encoder__drop=encoder_drop,
        cat_flags__flag_config=cat_flags_config,
        num_bin__mapping=num_bin_mapping,
    )

    # =========================
    # LIGHTGBM HYPERPARAMETERS
    # =========================
    lgb_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    pipeline.set_params(model=lgb.LGBMClassifier(**lgb_params))

    # =========================
    # CROSS-VALIDATION
    # =========================
    scores = cross_val_score(
        pipeline,
        X_raw,
        y_raw,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1,
    )

    return scores.mean()

### Run LightGBM Study

In [75]:
%%time

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress logs
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=200, show_progress_bar=False)

CPU times: user 2.59 s, sys: 975 ms, total: 3.56 s
Wall time: 3min 36s


In [76]:
print("Best ROC AUC:", study_lgb.best_value)
print("Best params:", study_lgb.best_params)

Best ROC AUC: 0.9368236257510172
Best params: {'poly_degree': 2, 'encoder_drop': None, 'cat_flags_idx': 1, 'num_bin_idx': 0, 'drop_cols': ['Tenure', 'HasCrCard', 'EstimatedSalary'], 'n_estimators': 375, 'max_depth': 3, 'learning_rate': 0.025030463251219333, 'num_leaves': 28, 'min_child_samples': 13, 'subsample': 0.5259828008944356, 'colsample_bytree': 0.7846454348813654, 'reg_alpha': 0.8285531440517132, 'reg_lambda': 4.648815918238638}


### Best Model Pipeline

In [79]:
def build_best_lgb_pipeline(study):
    best = study.best_params
    pipeline = clone(pipeline_rf)

    cat_flags_config = copy.deepcopy(cat_flags_config_all_options[best["cat_flags_idx"]])
    num_bin_mapping = copy.deepcopy(num_bin_mapping_all_options[best["num_bin_idx"]])

    pipeline.set_params(
        drop_cols__columns=best["drop_cols"],
        preprocessing__num__poly_features__degree=best["poly_degree"],
        preprocessing__cat__encoder__drop=best["encoder_drop"],
        cat_flags__flag_config=cat_flags_config,
        num_bin__mapping=num_bin_mapping,
    )

    lgb_params = {
        "n_estimators": best["n_estimators"],
        "max_depth": best["max_depth"],
        "learning_rate": best["learning_rate"],
        "num_leaves": best["num_leaves"],
        "min_child_samples": best["min_child_samples"],
        "subsample": best["subsample"],
        "colsample_bytree": best["colsample_bytree"],
        "reg_alpha": best["reg_alpha"],
        "reg_lambda": best["reg_lambda"],
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    pipeline.set_params(model=lgb.LGBMClassifier(**lgb_params))
    return pipeline

best_pipeline_lgb_opt = build_best_lgb_pipeline(study_lgb)
best_pipeline_lgb_opt.fit(X_raw, y_raw)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('num_flags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['Tenure', 'HasCrCard', ...]"
,flag_config,{}
,mapping,{}
,flag_config,"{'Geography': [{'drop_original': True, 'flag_name': 'IsGermany', 'value': 'Germany'}]}"
,config,{}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the Data

### Visualizations

In [77]:
optuna.visualization.plot_optimization_history(study_lgb)

In [78]:
optuna.visualization.plot_param_importances(study_lgb)

## Test Data Prediction

In [83]:
test_raw_df = pd.read_csv("downloads/test.csv")
test_raw_df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,15000,15594796.0,Chu,584.0,Germany,Male,30.0,2.0,146053.66,1.0,1.0,1.0,157891.86
1,15001,15642821.0,Mazzi,551.0,France,Male,39.0,5.0,0.00,2.0,1.0,1.0,67431.28
2,15002,15716284.0,Onyekachi,706.0,France,Male,43.0,8.0,0.00,2.0,1.0,0.0,156768.45
3,15003,15785078.0,Martin,717.0,Spain,Male,45.0,3.0,0.00,1.0,1.0,1.0,166909.87
4,15004,15662955.0,Kenechukwu,592.0,Spain,Male,43.0,8.0,0.00,2.0,1.0,1.0,143681.97


### Random Forest

In [93]:
test_pred_proba_rf = best_pipeline_rf_opt.predict_proba(test_raw_df)[:, 1]
test_raw_df["Exited_rf"] = test_pred_proba_rf.round(2)
test_raw_df["Exited_rf"].head()

0    0.39
1    0.08
2    0.27
3    0.86
4    0.19
Name: Exited_rf, dtype: float64

In [94]:
sample_submission_df = pd.read_csv("downloads/sample_submission.csv")
sample_submission_df["Exited"] = test_raw_df["Exited_rf"]
sample_submission_df.to_csv("downloads/submission_random_forest.csv", index=False)

### Adaboost

In [95]:
test_pred_proba_ada = best_pipeline_ada_opt.predict_proba(test_raw_df)[:, 1]
test_raw_df["Exited_ada"] = test_pred_proba_ada.round(2)
test_raw_df["Exited_ada"].head()

0    0.35
1    0.24
2    0.30
3    0.50
4    0.28
Name: Exited_ada, dtype: float64

In [96]:
sample_submission_df = pd.read_csv("downloads/sample_submission.csv")
sample_submission_df["Exited"] = test_raw_df["Exited_ada"]
sample_submission_df.to_csv("downloads/submission_adaboost.csv", index=False)

### XGBoost

In [97]:
test_pred_proba_xgb = best_pipeline_xgb_opt.predict_proba(test_raw_df)[:, 1]
test_raw_df["Exited_xgb"] = test_pred_proba_xgb.round(2)
test_raw_df["Exited_xgb"].head()

0    0.08
1    0.01
2    0.06
3    0.49
4    0.03
Name: Exited_xgb, dtype: float32

In [98]:
sample_submission_df = pd.read_csv("downloads/sample_submission.csv")
sample_submission_df["Exited"] = test_raw_df["Exited_xgb"]
sample_submission_df.to_csv("downloads/submission_xgboost.csv", index=False)

### LightGBM

In [99]:
test_pred_proba_xgb = best_pipeline_lgb_opt.predict_proba(test_raw_df)[:, 1]
test_raw_df["Exited_lgb"] = test_pred_proba_xgb.round(2)
test_raw_df["Exited_lgb"].head()

0    0.10
1    0.02
2    0.04
3    0.54
4    0.03
Name: Exited_lgb, dtype: float64

In [100]:
sample_submission_df = pd.read_csv("downloads/sample_submission.csv")
sample_submission_df["Exited"] = test_raw_df["Exited_lgb"]
sample_submission_df.to_csv("downloads/submission_lightgbm.csv", index=False)